# KLSE Index Movement Classification Using LSTM

This notebook demonstrates building a binary classifier that predicts whether the next trading-day closing price of the Kuala Lumpur Stock Exchange (KLSE) index will move up (1) or stay the same / move down (0).

High-level parts (so you can run and review step-by-step):

1. Setup & Imports
2. Data download
3. Data inspection & saving
4. Exploratory plots
5. Target creation
6. Feature selection & scaling
7. Sequence creation for LSTM
8. Train / Test split
9. Model building & Compile
10. Train The Model
11. Evaluation & visualization
12. Save results & next steps

Notes:

This notebook uses yfinance, pandas, scikit-learn and TensorFlow/Keras. Make sure these packages are installed in your environment.

# Step 1 — Setup & Imports

Import required libraries and set any global configuration. If you run this notebook from a different working directory, adjust paths in the save/load cells accordingly.

In [ ]:
import os
import pickle

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

from pathlib import Path

# Ensure folders are created at the project root
# If notebook is inside a "Notebook" folder, move one level up as project root
current_dir = Path.cwd()

if current_dir.name.lower() in ["notebook", "notebooks"]:
    project_root = current_dir.parent
else:
    project_root = current_dir

# Create output folders if they do not exist
os.makedirs(project_root / "Dataset", exist_ok=True)
os.makedirs(project_root / "Result", exist_ok=True)

print("Setup complete. Libraries imported and folders created.")

# Step 2 — Download KLSE data from Yahoo Finance

Define the KLSE ticker symbol and the date range to download. Modify the ticker or date range if you want data for other assets or periods.

In [ ]:
# Define the KLSE ticker
ticker = "^KLSE"

# Download historical data for the KLSE index
df = yf.download(
    ticker,
    start="2005-01-01",
    end="2026-01-01",
    interval="1d"
)

# Some yfinance versions return MultiIndex columns, even for a single ticker.
# Flatten the columns to keep standard names such as Open, High, Low, Close, and Volume.
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Sort by date and remove incomplete rows
df = df.sort_index()
df = df.dropna()

print("Data downloaded successfully.")
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Date range:", df.index.min(), "to", df.index.max())

# Step 3 — Data Inspection & Saving

Inspect the downloaded DataFrame to confirm successful retrieval and to get an initial sense of the data (rows, columns, missing values, basic statistics). Save a clean copy for reproducibility.

In [ ]:
# Display first 5 rows
print(df.head())

In [ ]:
# Display last 5 rows
print(df.tail())

In [ ]:
# Check dataset size
print("Dataset shape:", df.shape)

In [ ]:
# Check column names
print("Columns:")
print(df.columns)

In [ ]:
# Check dataset information
print("\nDataset information:")
print(df.info())

In [ ]:
# Check missing values
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
# Show basic statistical summary
print("\nStatistical summary:")
print(df.describe())

In [ ]:
# Save the raw downloaded dataset with Date preserved as a column
# Convert index min and max date into safe filename format
date_from = pd.to_datetime(df.index.min()).strftime("%Y-%m-%d")
date_to = pd.to_datetime(df.index.max()).strftime("%Y-%m-%d")
dataset_path=str(project_root / "Dataset" / f"KLSE_from_{date_from}_to_{date_to}_data.csv")
df.to_csv(dataset_path, index=True, index_label="Date")
print("Dataset saved successfully as {}".format(dataset_path))

# Step 4 — Exploratory Plots

Plot the Close price over time to visualize the long-term movement of the KLSE index. Use additional plots here for further EDA if desired.

In [ ]:
# Plot the closing price over time
plt.figure(figsize=(12, 6))
plt.plot(df.index, df['Close'], label='KLSE Closing Price')
plt.title('KLSE Closing Price Over Time')
plt.xlabel('Date')
plt.ylabel('Closing Price (MYR)')
plt.legend()
plt.grid()
plt.show()

# Step 5 — Target Creation

Create the binary target variable. The target is 1 when the next trading day's Close is greater than the current day's Close, otherwise 0. Drop the last row which has no next-day label.

In [ ]:
# Create target variable
# 1 = Next trading day closing price is higher than current day closing price
# 0 = Next trading day closing price is lower than or equal to current day closing price

# Create next-day closing price first
df["Next_Close"] = df["Close"].shift(-1)

# Create binary target variable
df["Target"] = (df["Next_Close"] > df["Close"]).astype(int)

# Remove the last row because it has no next trading day closing price
df = df.dropna(subset=["Next_Close"])

# Remove helper column after target creation
df = df.drop(columns=["Next_Close"])

# Check target result
print(df[["Close", "Target"]].head())
print(df[["Close", "Target"]].tail())

# Check class distribution
print(df["Target"].value_counts())

# Step 6 — Feature Selection & Scaling

Select input features for the model. Here we use common OHLCV features: Open, High, Low, Close and Volume. Then normalize the features to [0,1] with MinMaxScaler.

In [ ]:
# Select input features for LSTM model
features = ["Open", "High", "Low", "Close", "Volume"]

X = df[features]
y = df["Target"]

# Display selected features
print(X.head())
print(y.head())

# Check feature shape
print("Feature shape:", X.shape)
print("Target shape:", y.shape)

## Save UI Support Files

These files are required by the future UI application. The UI will use the same feature order and the latest 59 trading days to build a 60-day LSTM input sequence together with the user input.

In [ ]:
# Save the feature order for the UI application
features_path = project_root / "Result" / "KLSE_features.pkl"

with open(features_path, "wb") as file:
    pickle.dump(features, file)

print(f"Feature list saved successfully as {features_path}")

# Save the latest 29 trading days for UI prediction
# UI logic: latest 29 historical trading days + 1 user input day = 30-day LSTM input sequence
latest_29_days = df[features].tail(29)
latest_29_days.to_csv(project_root / "Result" / "KLSE_latest_29_days.csv", index=False)

print("Latest 29 trading days saved successfully as {}".format(project_root / "Result" / "KLSE_latest_29_days.csv"))

# Save the latest 59 trading days for UI prediction
# UI logic: latest 59 historical trading days + 1 user input day = 60-day LSTM input sequence
latest_59_days = df[features].tail(59)
latest_59_days.to_csv(project_root / "Result" / "KLSE_latest_59_days.csv", index=False)

print("Latest 59 trading days saved successfully as {}".format(project_root / "Result" / "KLSE_latest_59_days.csv"))

# Save the latest 89 trading days for UI prediction
# UI logic: latest 89 historical trading days + 1 user input day = 90-day LSTM input sequence
latest_89_days = df[features].tail(89)
latest_89_days.to_csv(project_root / "Result" / "KLSE_latest_89_days.csv", index=False)

print("Latest 89 trading days saved successfully as {}".format(project_root / "Result" / "KLSE_latest_89_days.csv"))

In [ ]:
# Normalise the input features into range 0 to 1
scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)

# Check scaled data
print(X_scaled[:5])
print("Scaled feature shape:", X_scaled.shape)


## Save Trained Scaler

The scaler is saved because the UI application must apply the same feature scaling used during model training.

In [ ]:
# Save the trained MinMaxScaler for the UI application
features_path = project_root / "Result" /  "KLSE_scaler.pkl"

with open(features_path, "wb") as file:
    pickle.dump(scaler, file)

print("Scaler saved successfully as {}".format(features_path))

# Step 7 — Sequence Creation for LSTM

Create time-series sequences for LSTM input. We use a lookback window of 60 trading days (time_steps = 60). The function below converts the scaled feature matrix and labels into (samples, time_steps, features) and corresponding labels.

In [ ]:
# Function to create LSTM sequences
def create_sequences(X, y, time_steps):
    Xs, ys = [], []

    for i in range(time_steps, len(X)):
        Xs.append(X[i-time_steps:i])
        ys.append(y.iloc[i])

    return np.array(Xs), np.array(ys)


# Support multiple sequence lengths
time_steps_list = [30, 60, 90]

sequence_data = {}

for time_steps in time_steps_list:
    X_seq, y_seq = create_sequences(X_scaled, y, time_steps)

    sequence_data[time_steps] = {
        "X_seq": X_seq,
        "y_seq": y_seq
    }

    print(f"{time_steps}-day sequence created successfully")
    print("X sequence shape:", X_seq.shape)
    print("y sequence shape:", y_seq.shape)
    print("-" * 50)

# Step 8 — Train / Test Split

Split sequences into training and testing sets. We use an 80/20 split (no random shuffle to preserve time order). This is important for time-series forecasting/classification to avoid look-ahead bias.

In [ ]:
# Split data into training and testing sets for 30, 60, and 90 days
# 80% training data, 20% testing data

split_data = {}

for time_steps in time_steps_list:
    X_seq = sequence_data[time_steps]["X_seq"]
    y_seq = sequence_data[time_steps]["y_seq"]

    train_size = int(len(X_seq) * 0.8)

    X_train = X_seq[:train_size]
    X_test = X_seq[train_size:]

    y_train = y_seq[:train_size]
    y_test = y_seq[train_size:]

    split_data[time_steps] = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test
    }

    print(f"{time_steps}-day train-test split completed")
    print("X_train shape:", X_train.shape)
    print("X_test shape:", X_test.shape)
    print("y_train shape:", y_train.shape)
    print("y_test shape:", y_test.shape)
    print("-" * 50)

# Step 9 — Model Building

Build an LSTM-based Keras sequential model for binary classification. The architecture below is a simple baseline: two LSTM layers with dropout and a sigmoid output neuron.

In [ ]:
# Function to build LSTM model
def build_lstm_model(input_shape):
    model = Sequential()

    # First LSTM layer
    model.add(LSTM(
        units=64,
        return_sequences=True,
        input_shape=input_shape
    ))
    model.add(Dropout(0.2))

    # Second LSTM layer
    model.add(LSTM(
        units=32,
        return_sequences=False
    ))
    model.add(Dropout(0.2))

    # Output layer for binary classification
    model.add(Dense(1, activation="sigmoid"))

    # Compile model
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

# Build models for 30, 60, and 90 trading days

models = {}

for time_steps in time_steps_list:
    X_train = split_data[time_steps]["X_train"]

    input_shape = (X_train.shape[1], X_train.shape[2])

    model = build_lstm_model(input_shape)

    models[time_steps] = model

    print(f"{time_steps}-day LSTM model created successfully")
    print(f"Input shape: {input_shape}")
    model.summary()
    print("-" * 50)

# Step 10 — Train The Model

Compile the model with Adam optimizer and binary cross-entropy loss. Accuracy is used as a simple performance metric; consider adding AUC for imbalanced datasets. Then train the model.

In [ ]:
# Train the LSTM model
histories = {}

for time_steps in time_steps_list:
    model = models[time_steps]

    X_train = split_data[time_steps]["X_train"]
    y_train = split_data[time_steps]["y_train"]

    print(f"Training {time_steps}-day LSTM model...")

    history = model.fit(
        X_train,
        y_train,
        epochs=30,
        batch_size=32,
        validation_split=0.2,
        verbose=1
    )

    histories[time_steps] = history

    print(f"{time_steps}-day model training completed")
    print("-" * 50)

# Step 11 — Evaluation & Visualization

Plot training/validation accuracy and loss to inspect training behavior for signs of overfitting or underfitting. After training, make predictions on the test set and calculate evaluation metrics.

In [ ]:
# Save training accuracy and loss graphs for each sequence length

for time_steps in time_steps_list:
    history = histories[time_steps]

    # Accuracy graph
    plt.figure(figsize=(8, 5))
    plt.plot(history.history["accuracy"], label="Training Accuracy")
    plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
    plt.title(f"LSTM Model Accuracy ({time_steps}-Day Sequence)")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    accuracy_path = project_root / "Result" / f"KLSE_LSTM_accuracy_{time_steps}.png"
    plt.savefig(accuracy_path, bbox_inches="tight")
    plt.show()

    # Loss graph
    plt.figure(figsize=(8, 5))
    plt.plot(history.history["loss"], label="Training Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.title(f"LSTM Model Loss ({time_steps}-Day Sequence)")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    loss_path = project_root / "Result" / f"KLSE_LSTM_loss_{time_steps}.png"
    plt.savefig(loss_path, bbox_inches="tight")
    plt.show()

    print(f"{time_steps}-day accuracy graph saved as {accuracy_path}")
    print(f"{time_steps}-day loss graph saved as {loss_path}")

Make predictions on the test set and convert predicted probabilities to class labels using a 0.5 threshold. You can change the threshold to tune precision/recall trade-offs.

In [ ]:
# Make predictions for each sequence length
# Convert predicted probabilities to class labels using 0.5 threshold

prediction_results = {}

for time_steps in time_steps_list:
    model = models[time_steps]

    X_test = split_data[time_steps]["X_test"]
    y_test = split_data[time_steps]["y_test"]

    # Make predictions on the test dataset
    y_pred_prob = model.predict(X_test)

    # Convert prediction probabilities into class labels
    y_pred = (y_pred_prob > 0.5).astype(int)

    # Flatten predicted labels for evaluation
    y_pred = y_pred.flatten()

    prediction_results[time_steps] = {
        "y_test": y_test,
        "y_pred_prob": y_pred_prob,
        "y_pred": y_pred
    }

    print(f"{time_steps}-day model prediction completed")
    print("First 10 prediction probabilities:")
    print(y_pred_prob[:10])

    print("\nFirst 10 predicted classes:")
    print(y_pred[:10])

    print("\nFirst 10 actual classes:")
    print(y_test[:10])
    print("-" * 50)

Prepare predictions and evaluate model performance using accuracy, precision, recall and F1. The notebook uses zero_division=0 in metrics to avoid exceptions for empty predicted classes.

In [ ]:
# Evaluate models for 30, 60, and 90 days

evaluation_results = {}
confusion_matrices = {}

for time_steps in time_steps_list:
    y_test = prediction_results[time_steps]["y_test"]
    y_pred = prediction_results[time_steps]["y_pred"]

    # Ensure predicted labels are flattened
    y_pred = y_pred.flatten()

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)

    # Store results
    evaluation_results[time_steps] = {
        "Sequence_Days": time_steps,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1_score": f1
    }

    confusion_matrices[time_steps] = cm

    # Print results
    print(f"{time_steps}-Day Model Evaluation Results")
    print("-" * 40)
    print("y_test shape:", y_test.shape)
    print("y_pred shape:", y_pred.shape)
    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1-score:", f1)

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    print("\nConfusion Matrix:")
    print(cm)
    print("=" * 60)

In [ ]:
# Convert evaluation results to DataFrame
evaluation_df = pd.DataFrame(evaluation_results.values())

# Display comparison table
print("LSTM Model Evaluation Comparison")
print(evaluation_df)

# Save evaluation comparison result
evaluation_path = project_root / "Result" / "KLSE_LSTM_sequence_comparison.csv"
evaluation_df.to_csv(evaluation_path, index=False)

print(f"Evaluation comparison saved successfully as {evaluation_path}")

Visualize the confusion matrix and compare actual vs predicted movements for a subset of test samples.

In [ ]:
# Visualize confusion matrix and compare actual vs predicted movements
# for 30-day, 60-day, and 90-day LSTM models

sample_size = 100

for time_steps in time_steps_list:
    y_test = prediction_results[time_steps]["y_test"]
    y_pred = prediction_results[time_steps]["y_pred"]
    cm = confusion_matrices[time_steps]

    # -----------------------------
    # Confusion Matrix Heatmap
    # -----------------------------
    plt.figure(figsize=(6, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Down/Unchanged", "Up"],
        yticklabels=["Down/Unchanged", "Up"]
    )

    plt.title(f"Confusion Matrix ({time_steps}-Day Sequence)")
    plt.xlabel("Predicted Class")
    plt.ylabel("Actual Class")
    plt.tight_layout()

    cm_plot_path = project_root / "Result" / f"KLSE_LSTM_confusion_matrix_{time_steps}.png"
    plt.savefig(cm_plot_path, bbox_inches="tight")
    plt.show()

    print(f"{time_steps}-day confusion matrix plot saved as {cm_plot_path}")

    # -----------------------------
    # Actual vs Predicted Movement
    # -----------------------------
    plt.figure(figsize=(12, 5))
    plt.plot(y_test[:sample_size], label="Actual Movement", marker="o")
    plt.plot(y_pred[:sample_size], label="Predicted Movement", marker="x")

    plt.title(f"Actual vs Predicted KLSE Movement ({time_steps}-Day Sequence)")
    plt.xlabel("Test Sample")
    plt.ylabel("Movement Class")
    plt.yticks([0, 1], ["Down/Unchanged", "Up"])
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    actual_pred_plot_path = project_root / "Result" / f"KLSE_LSTM_actual_vs_predicted_{time_steps}.png"
    plt.savefig(actual_pred_plot_path, bbox_inches="tight")
    plt.show()

    print(f"{time_steps}-day actual vs predicted plot saved as {actual_pred_plot_path}")
    print("-" * 60)


In [ ]:
# Save actual vs predicted results for each sequence length

for time_steps in time_steps_list:
    y_test = prediction_results[time_steps]["y_test"]
    y_pred = prediction_results[time_steps]["y_pred"]
    y_pred_prob = prediction_results[time_steps]["y_pred_prob"].flatten()

    prediction_df = pd.DataFrame({
        "Actual": y_test,
        "Predicted": y_pred,
        "Upward_Probability": y_pred_prob
    })

    prediction_csv_path = project_root / "Result" / f"KLSE_LSTM_prediction_results_{time_steps}.csv"
    prediction_df.to_csv(prediction_csv_path, index=False)

    print(f"{time_steps}-day prediction results saved as {prediction_csv_path}")

# Step 12 — Save Results & Next Steps

Save the trained model and evaluation results (evaluation metrics, prediction results, confusion matrix) to the project's Result folder so outputs can be reviewed and reused. Also includes suggestions for next steps.

In [ ]:
# Save trained models and evaluation outputs to the Result folder
# This includes model files, evaluation metrics, prediction results, and confusion matrices

# Save overall evaluation comparison table
evaluation_df = pd.DataFrame(evaluation_results.values())

evaluation_comparison_path = project_root / "Result" / "KLSE_LSTM_sequence_comparison.csv"
evaluation_df.to_csv(evaluation_comparison_path, index=False)

print(f"Evaluation comparison saved successfully as {evaluation_comparison_path}")

# Save outputs for each sequence length
for time_steps in time_steps_list:
    model = models[time_steps]

    y_test = prediction_results[time_steps]["y_test"]
    y_pred = prediction_results[time_steps]["y_pred"]
    y_pred_prob = prediction_results[time_steps]["y_pred_prob"].flatten()
    cm = confusion_matrices[time_steps]

    # -----------------------------
    # Save trained model
    # -----------------------------
    model_h5_path = project_root / "Result" / f"KLSE_LSTM_model_{time_steps}.h5"
    model_keras_path = project_root / "Result" / f"KLSE_LSTM_model_{time_steps}.keras"

    model.save(model_h5_path)
    model.save(model_keras_path)

    # -----------------------------
    # Save evaluation metrics
    # -----------------------------
    metrics_df = pd.DataFrame([evaluation_results[time_steps]])

    metrics_path = project_root / "Result" / f"KLSE_LSTM_evaluation_results_{time_steps}.csv"
    metrics_df.to_csv(metrics_path, index=False)

    # -----------------------------
    # Save prediction results
    # -----------------------------
    prediction_df = pd.DataFrame({
        "Actual": y_test,
        "Predicted": y_pred,
        "Upward_Probability": y_pred_prob
    })

    prediction_path = project_root / "Result" / f"KLSE_LSTM_prediction_results_{time_steps}.csv"
    prediction_df.to_csv(prediction_path, index=False)

    # -----------------------------
    # Save confusion matrix
    # -----------------------------
    confusion_matrix_df = pd.DataFrame(
        cm,
        index=["Actual Down/Unchanged", "Actual Up"],
        columns=["Predicted Down/Unchanged", "Predicted Up"]
    )

    confusion_matrix_path = project_root / "Result" / f"KLSE_LSTM_confusion_matrix_{time_steps}.csv"
    confusion_matrix_df.to_csv(confusion_matrix_path)

    print(f"{time_steps}-day model and results saved successfully.")
    print(f"Model H5: {model_h5_path}")
    print(f"Model Keras: {model_keras_path}")
    print(f"Metrics: {metrics_path}")
    print(f"Predictions: {prediction_path}")
    print(f"Confusion Matrix: {confusion_matrix_path}")
    print("-" * 60)

# Save 60-day model as default model for compatibility with UI or earlier code
default_time_steps = 60

default_model_h5_path = project_root / "Result" / "KLSE_LSTM_model.h5"
default_model_keras_path = project_root / "Result" / "KLSE_LSTM_model.keras"

models[default_time_steps].save(default_model_h5_path)
models[default_time_steps].save(default_model_keras_path)

print("Default 60-day model also saved for compatibility.")
print(f"Default H5 model: {default_model_h5_path}")
print(f"Default Keras model: {default_model_keras_path}")


In [ ]:
# Save latest historical records for UI application
# User input will be added as the final day to form complete sequences

latest_days_config = {
    30: 29,
    60: 59,
    90: 89
}

for time_steps, latest_days in latest_days_config.items():
    latest_data = df[features].tail(latest_days)

    latest_data_path = project_root / "Result" / f"KLSE_latest_{latest_days}_days.csv"
    latest_data.to_csv(latest_data_path, index=False)

    print(f"Latest {latest_days} days saved for {time_steps}-day UI input as {latest_data_path}")